# Feature Selection & Correlation

This notebook loads the raw processed data, cleans it, splits it into train/validation/test, and performs correlation-based feature selection. The final selected feature set (26 features) is saved to disk for use in the modeling notebook (`02_random_forest_model.ipynb`).

In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
   "../../data/processed/store_day.csv"
)
df.head()

,Location,Date,Gross_Sales,Discounts,Net_Sales,Tax,Total_Collected,Gross_Profit,Total_Cost,Transactions,...,Net_Sales_Lag_7,Transactions_Lag_1,Transactions_Lag_7,Profit_Rolling_7D,Net_Sales_Rolling_7D,Transactions_Rolling_7D,Profit_Growth_7D,Net_Sales_Growth_7D,Transactions_Growth_7D,Future_7D_Gross_Profit
0,Store 01 - Austin,2024-01-01,456.0,-37.15,418.85,34.56,453.41,212.82,206.03,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Store 01 - Austin,2024-01-02,238.0,-5.90,232.10,19.15,251.25,130.46,101.64,4.0,...,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Store 01 - Austin,2024-01-03,296.0,-40.72,255.28,21.07,276.35,136.06,119.22,2.0,...,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Store 01 - Austin,2024-01-04,478.0,-12.54,465.46,38.42,503.88,281.20,184.26,9.0,...,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Store 01 - Austin,2024-01-05,902.0,-59.07,842.93,69.54,912.47,487.54,355.39,16.0,...,NaN,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.columns

Index(['Location', 'Date', 'Gross_Sales', 'Discounts', 'Net_Sales', 'Tax',
       'Total_Collected', 'Gross_Profit', 'Total_Cost', 'Transactions',
       'Units', 'Refund_Lines', 'Discounted_Lines',
       'Identified_Customer_Lines', 'Unique_SKUs', 'Store_Closed',
       'Profit_Margin', 'Discount_Rate', 'Avg_Transaction_Value',
       'Units_Per_Transaction', 'Profit_Lag_1', 'Profit_Lag_7',
       'Net_Sales_Lag_1', 'Net_Sales_Lag_7', 'Transactions_Lag_1',
       'Transactions_Lag_7', 'Profit_Rolling_7D', 'Net_Sales_Rolling_7D',
       'Transactions_Rolling_7D', 'Profit_Growth_7D', 'Net_Sales_Growth_7D',
       'Transactions_Growth_7D', 'Future_7D_Gross_Profit'],
      dtype='object')

In [4]:
model_data = df.copy()

model_data = model_data.dropna(
    subset=["Future_7D_Gross_Profit"]
)

model_data.shape

(21720, 33)

In [5]:
closed_day_cols = [
    "Profit_Margin",
    "Discount_Rate",
    "Avg_Transaction_Value",
    "Units_Per_Transaction",
    "Profit_Growth_7D",
    "Net_Sales_Growth_7D",
    "Transactions_Growth_7D"
]

model_data.loc[
    model_data["Store_Closed"] == 1,
    closed_day_cols
] = 0

In [6]:
growth_cols = [
    "Profit_Growth_7D",
    "Net_Sales_Growth_7D",
    "Transactions_Growth_7D"
]

model_data[growth_cols] = model_data[growth_cols].fillna(0)

In [7]:
model_data.isna().sum()[model_data.isna().sum() > 0]

Profit_Lag_7          30
Net_Sales_Lag_7       30
Transactions_Lag_7    30
dtype: int64

In [8]:
model_data = (
    model_data
    .sort_values(["Location", "Date"])
    .groupby("Location")
    .apply(lambda x: x.iloc[7:])
    .reset_index(drop=True)
)

C:\Users\ShehabYousef\AppData\Local\Temp\ipykernel_9228\1452950607.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[7:])


In [9]:
train = model_data[model_data["Date"] <= "2025-06-30"].copy()

validation = model_data[
    (model_data["Date"] >= "2025-07-01") &
    (model_data["Date"] <= "2025-09-30")
].copy()

test = model_data[model_data["Date"] >= "2025-10-01"].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (16020, 33)
Validation: (2760, 33)
Test: (2730, 33)


In [10]:
target = "Future_7D_Gross_Profit"

X_train = train.drop(columns=[target])
y_train = train[target]

X_validation = validation.drop(columns=[target])
y_validation = validation[target]

X_test = test.drop(columns=[target])
y_test = test[target]

## Feature Selection

In [11]:
feature_candidates = [
    "Gross_Sales",
    "Discounts",
    "Net_Sales",
    "Tax",
    "Total_Collected",
    "Gross_Profit",
    "Total_Cost",
    "Transactions",
    "Units",
    "Refund_Lines",
    "Discounted_Lines",
    "Identified_Customer_Lines",
    "Unique_SKUs",
    "Store_Closed",
    "Profit_Margin",
    "Discount_Rate",
    "Avg_Transaction_Value",
    "Units_Per_Transaction",
    "Profit_Lag_1",
    "Profit_Lag_7",
    "Net_Sales_Lag_1",
    "Net_Sales_Lag_7",
    "Transactions_Lag_1",
    "Transactions_Lag_7",
    "Profit_Rolling_7D",
    "Net_Sales_Rolling_7D",
    "Transactions_Rolling_7D",
    "Profit_Growth_7D",
    "Net_Sales_Growth_7D",
    "Transactions_Growth_7D"
]

X_train_candidates = X_train[feature_candidates]
X_validation_candidates = X_validation[feature_candidates]
X_test_candidates = X_test[feature_candidates]

In [12]:
correlation = (
    X_train_candidates
    .corrwith(y_train)
    .sort_values(key=abs, ascending=False)
)

correlation

Profit_Rolling_7D            0.992165
Net_Sales_Rolling_7D         0.990217
Transactions_Rolling_7D      0.978720
Profit_Lag_1                 0.790307
Gross_Profit                 0.789325
Transactions_Lag_1           0.786967
Transactions                 0.786430
Net_Sales_Lag_1              0.783981
Tax                          0.783070
Total_Collected              0.783068
Net_Sales                    0.783067
Units                        0.781999
Unique_SKUs                  0.778335
Gross_Sales                  0.776393
Total_Cost                   0.770607
Profit_Lag_7                 0.740052
Transactions_Lag_7           0.732415
Net_Sales_Lag_7              0.729477
Identified_Customer_Lines    0.712223
Discounted_Lines             0.658336
Discounts                   -0.516393
Refund_Lines                 0.315884
Profit_Margin               -0.100029
Discount_Rate                0.078301
Store_Closed                 0.047092
Profit_Growth_7D            -0.023048
Net_Sales_Gr

In [13]:
corr_matrix = X_train_candidates.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs[high_corr_pairs >= 0.90]

Net_Sales             Total_Collected              1.000000
Tax                   Total_Collected              1.000000
Net_Sales             Tax                          1.000000
Profit_Rolling_7D     Net_Sales_Rolling_7D         0.998723
Gross_Sales           Total_Cost                   0.998507
                      Net_Sales                    0.998057
                      Total_Collected              0.998057
                      Tax                          0.998056
Tax                   Gross_Profit                 0.997556
Total_Collected       Gross_Profit                 0.997555
Net_Sales             Gross_Profit                 0.997555
Profit_Lag_1          Net_Sales_Lag_1              0.997554
Profit_Lag_7          Net_Sales_Lag_7              0.997550
Net_Sales             Total_Cost                   0.996525
Total_Collected       Total_Cost                   0.996525
Tax                   Total_Cost                   0.996524
Gross_Sales           Gross_Profit      

In [14]:
# Final feature set after correlation-based reduction

selected_features = [
    col for col in feature_candidates
    if col not in ["Gross_Sales", "Tax", "Total_Collected", "Total_Cost"]
]

X_train_final = X_train[selected_features]
X_validation_final = X_validation[selected_features]
X_test_final = X_test[selected_features]

print("Number of selected features:", len(selected_features))
print("Features:")
print(selected_features)

print("\nShapes:")
print("Train:", X_train_final.shape)
print("Validation:", X_validation_final.shape)
print("Test:", X_test_final.shape)

Number of selected features: 26
Features:
['Discounts', 'Net_Sales', 'Gross_Profit', 'Transactions', 'Units', 'Refund_Lines', 'Discounted_Lines', 'Identified_Customer_Lines', 'Unique_SKUs', 'Store_Closed', 'Profit_Margin', 'Discount_Rate', 'Avg_Transaction_Value', 'Units_Per_Transaction', 'Profit_Lag_1', 'Profit_Lag_7', 'Net_Sales_Lag_1', 'Net_Sales_Lag_7', 'Transactions_Lag_1', 'Transactions_Lag_7', 'Profit_Rolling_7D', 'Net_Sales_Rolling_7D', 'Transactions_Rolling_7D', 'Profit_Growth_7D', 'Net_Sales_Growth_7D', 'Transactions_Growth_7D']

Shapes:
Train: (16020, 26)
Validation: (2760, 26)
Test: (2730, 26)


## Save Outputs

Save the final selected-feature datasets and the feature list so the modeling notebook can load them directly without repeating the cleaning/selection steps.

In [15]:
import os

output_dir = "../../data/processed/feature_selection"
os.makedirs(output_dir, exist_ok=True)

# Save feature matrices + targets
X_train_final.assign(**{target: y_train}).to_csv(f"{output_dir}/train_final.csv", index=False)
X_validation_final.assign(**{target: y_validation}).to_csv(f"{output_dir}/validation_final.csv", index=False)
X_test_final.assign(**{target: y_test}).to_csv(f"{output_dir}/test_final.csv", index=False)

# Save the selected feature list + target name
with open(f"{output_dir}/selected_features.json", "w") as f:
    json.dump({"target": target, "selected_features": selected_features}, f, indent=2)

print("Saved train/validation/test sets and selected_features.json to:", output_dir)

Saved train/validation/test sets and selected_features.json to: ../../data/processed/feature_selection
